# Retrain "CNN Advance" (DenseNet121) — Colab / T4 GPU

This notebook trains a **real** DenseNet121 classifier to replace `cnn_advance.pth`, for which no original training script existed anywhere in the project (only inference code in `model_loader.py`/`verify_densenet.py`).

Preprocessing matches `models/backend/model_loader.py`'s PyTorch inference path exactly (Resize 224x224, ToTensor, **no ImageNet normalization**) so the resulting weights are a drop-in replacement for `cnn_advance.pth` — no backend code changes needed.

## Before running
1. Zip your `train` and `test` folders separately (e.g. right-click `DATA\train` → Send to → Compressed folder, and the same for `DATA\test`). This gives `train.zip` and `test.zip`.
2. Upload both to your Google Drive, e.g. into `My Drive/DATASET/train.zip` and `My Drive/DATASET/test.zip`.
3. In Colab: **Runtime → Change runtime type → T4 GPU**.
4. Run all cells top to bottom. **Cell 2 now prints the resolved class folders and hard-stops with an assertion error if it doesn't find exactly 2+ real classes** — do not proceed past a failed assertion, it means the path is still wrong. (This is the fix for the bug that broke the previous run: `train.zip`/`test.zip` each contain a wrapper folder, e.g. `train/fracture/...`, and pointing `ImageFolder` at the wrapper folder itself collapses everything into one fake class called `"train"`/`"test"` — which is exactly what happened and is why that run showed a meaningless 100% accuracy.)
5. At the end, download `cnn_advance_retrained.pth` and `classification_report_CNN_Advance_RETRAINED.txt` and drop them into `models/extra/` (and copy the `.pth` into `models/backend/models/` and `models/backend/new model/`, replacing/renaming to `cnn_advance.pth` if you want the backend to use it).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT THIS to match where you uploaded train.zip and test.zip in your Drive
DRIVE_TRAIN_ZIP_PATH = '/content/drive/MyDrive/DATASET/train.zip'
DRIVE_TEST_ZIP_PATH = '/content/drive/MyDrive/DATASET/test.zip'

import os, zipfile

assert os.path.exists(DRIVE_TRAIN_ZIP_PATH), f"Train zip not found at {DRIVE_TRAIN_ZIP_PATH} - fix the path above"
assert os.path.exists(DRIVE_TEST_ZIP_PATH), f"Test zip not found at {DRIVE_TEST_ZIP_PATH} - fix the path above"

TRAIN_EXTRACT_ROOT = '/content/train_data_extracted'
TEST_EXTRACT_ROOT = '/content/test_data_extracted'
os.makedirs(TRAIN_EXTRACT_ROOT, exist_ok=True)
os.makedirs(TEST_EXTRACT_ROOT, exist_ok=True)

with zipfile.ZipFile(DRIVE_TRAIN_ZIP_PATH, 'r') as z:
    z.extractall(TRAIN_EXTRACT_ROOT)
with zipfile.ZipFile(DRIVE_TEST_ZIP_PATH, 'r') as z:
    z.extractall(TEST_EXTRACT_ROOT)

def resolve_class_dir(path, label):
    """Descend past any wrapper folders (e.g. zip containing a single top-level
    'train/' or 'test/' folder) until we hit the level with multiple class subfolders.
    Fails loudly instead of silently training on a broken single-class dataset."""
    seen = [path]
    for _ in range(5):
        entries = os.listdir(path)
        subdirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
        if len(subdirs) == 1:
            path = os.path.join(path, subdirs[0])
            seen.append(path)
            continue
        break
    entries = os.listdir(path)
    subdirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    print(f'[{label}] descended through: {seen}')
    print(f'[{label}] resolved dir: {path} -> class folders: {subdirs}')
    assert len(subdirs) >= 2, (
        f"[{label}] Expected >=2 class folders at {path}, found {subdirs}. "
        "Fix your zip structure before continuing - do NOT run training on this."
    )
    return path

TRAIN_DIR = resolve_class_dir(TRAIN_EXTRACT_ROOT, 'TRAIN')
TEST_DIR = resolve_class_dir(TEST_EXTRACT_ROOT, 'TEST')
print('Final TRAIN_DIR:', TRAIN_DIR)
print('Final TEST_DIR:', TEST_DIR)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import time, json
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

torch.manual_seed(42)

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-4
VAL_SPLIT = 0.1

# Matches model_loader.py's PyTorch preprocessing exactly (no Normalize step).
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
def build_model():
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.features.denseblock4.parameters():
        param.requires_grad = True
    for param in model.features.norm5.parameters():
        param.requires_grad = True
    model.classifier = nn.Linear(model.classifier.in_features, 1)
    return model.to(DEVICE)

full_train = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
print('Classes (alphabetical -> index):', full_train.class_to_idx)
assert len(full_train.class_to_idx) == 2, (
    f"Expected exactly 2 classes in TRAIN_DIR, got {full_train.class_to_idx}. "
    "Do not proceed - fix TRAIN_DIR in the previous cell."
)

n_val = int(len(full_train) * VAL_SPLIT)
n_train = len(full_train) - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full_train, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)
val_ds.dataset = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)

test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
print('Test classes (alphabetical -> index):', test_ds.class_to_idx)
assert len(test_ds.class_to_idx) == 2, (
    f"Expected exactly 2 classes in TEST_DIR, got {test_ds.class_to_idx}. "
    "Do not proceed - fix TEST_DIR in the previous cell."
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

In [ ]:
model = build_model()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
best_state = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.float().unsqueeze(1).to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            v_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            v_correct += (preds == labels).sum().item()
            v_total += labels.size(0)
    val_loss = v_loss / v_total
    val_acc = v_correct / v_total
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    print(f'[EPOCH {epoch}/{EPOCHS}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
          f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({elapsed:.1f}s)')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

with open('/content/cnn_advance_history.json', 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
model.eval()
all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        probs = torch.sigmoid(outputs).squeeze(1)
        preds = (probs >= 0.5).float()
        all_labels.extend(labels.numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

class_names = [k for k, v in sorted(test_ds.class_to_idx.items(), key=lambda x: x[1])]
report = classification_report(all_labels, all_preds, target_names=class_names, digits=2)
cm = confusion_matrix(all_labels, all_preds)
try:
    auc = roc_auc_score(all_labels, all_probs)
except ValueError:
    auc = float('nan')

print(report)
print('Confusion matrix:\n', cm)
print(f'ROC-AUC: {auc:.4f}')

with open('/content/classification_report_CNN_Advance_RETRAINED.txt', 'w') as f:
    f.write(f'Classification Report for CNN_Advance (retrained DenseNet121, {EPOCHS} epochs, T4 GPU)\n')
    f.write('=' * 60 + '\n')
    f.write(report)
    f.write(f'\nROC-AUC: {auc:.4f}\n')
    f.write(f'\nConfusion matrix (rows=true, cols=pred, order={class_names}):\n{cm}\n')
    f.write(f'\nClass index mapping: {test_ds.class_to_idx}\n')

torch.save(model.state_dict(), '/content/cnn_advance_retrained.pth')
print('Saved to /content/')

In [ ]:
from google.colab import files
files.download('/content/cnn_advance_retrained.pth')
files.download('/content/classification_report_CNN_Advance_RETRAINED.txt')
files.download('/content/cnn_advance_history.json')